# Chibani Lotfi AI — Fast Google Colab Worker

One Render service controls the queue; Colab runs MoneyPrinterTurbo and uploads only the final MP4 to Google Drive.


In [ ]:
import os, pathlib
from google.colab import drive, auth, userdata
if not pathlib.Path("/content/Chibani-Lotfi-AI").exists():
    !git clone https://github.com/koreaone10-del/Chibani-Lotfi-AI.git /content/Chibani-Lotfi-AI
if not pathlib.Path("/content/MoneyPrinterTurbo").exists():
    !git clone https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo
%cd /content/MoneyPrinterTurbo
!git checkout 436b0e9


In [ ]:
drive.mount('/content/drive')
auth.authenticate_user()
!pip -q install -r /content/Chibani-Lotfi-AI/worker/requirements.txt
!pip -q install -r /content/MoneyPrinterTurbo/requirements.txt


In [ ]:
def secret(name, default=""):
    try:
        return userdata.get(name)
    except Exception:
        return os.getenv(name, default)
CONTROL_API_URL=secret("CONTROL_API_URL")
WORKER_API_KEY=secret("WORKER_API_KEY")
PEXELS_API_KEY=secret("PEXELS_API_KEY")
GEMINI_API_KEY=secret("GEMINI_API_KEY")
assert CONTROL_API_URL and WORKER_API_KEY, "Add CONTROL_API_URL and WORKER_API_KEY to Colab Secrets."
config = "\n".join([
    'log_level = "INFO"',
    'listen_host = "127.0.0.1"',
    'listen_port = 8080',
    'edge_tts_timeout = 30',
    'tls_verify = true',
    'video_source = "pexels"',
    "pexels_api_keys = " + repr([PEXELS_API_KEY]),
    'llm_provider = "gemini"',
    "gemini_api_key = " + repr(GEMINI_API_KEY),
    'subtitle_provider = "edge"',
    'endpoint = ""',
    'max_concurrent_tasks = 1',
    'max_queued_tasks = 3',
    ''
])
pathlib.Path('/content/MoneyPrinterTurbo/config.toml').write_text(config, encoding='utf-8')
os.environ.update({"CONTROL_API_URL":CONTROL_API_URL,"WORKER_API_KEY":WORKER_API_KEY,"MPT_DIR":"/content/MoneyPrinterTurbo","MPT_PORT":"8080","POLL_SECONDS":"2","MPT_VOICE_AR":"ar-DZ-IsmaelNeural-Male","MPT_FONT_NAME":"STHeitiMedium.ttc"})
print("Ready")


In [ ]:
%cd /content/Chibani-Lotfi-AI/worker
!python worker.py
